# AI-Based Business Location Selector: Supervised Model Benchmarking
**Objective:** Train, evaluate, and compare Linear Regression, Decision Tree, and Random Forest Regressors for commercial demand prediction.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

data_path = Path("../data/business_location_synthetic_dataset.csv")
if not data_path.exists():
    data_path = Path("business_location_synthetic_dataset.csv")
df = pd.read_csv(data_path)

# Feature selection
feature_cols = [
    "monthly_rent", "population", "competitor_count", "daily_footfall",
    "accessibility_score", "parking_score", "public_transport_score"
]
X = df[feature_cols]
y = df["target_demand"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

## 1. Train Models and Evaluate Metrics (MAE, RMSE, R²)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    cv = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="r2").mean()
    
    results.append({
        "Model": name,
        "Test R²": r2,
        "5-Fold CV R²": cv,
        "Test MAE": mae,
        "Test RMSE": rmse
    })

df_results = pd.DataFrame(results)
display(df_results)

## 2. Feature Importance from Random Forest

In [ ]:
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(x=importances.values, y=importances.index, palette="mako")
plt.title("Random Forest Feature Importances for Demand Prediction")
plt.xlabel("Mean Decrease in Impurity (Importance)")
plt.show()